In [ ]:
import requests
import pandas as pd
import time

pd.set_option('display.max_columns', None)

# --- Configuration ---
API_KEY   = 'XXXXXX'
BASE_URL  = 'https://api.shovels.ai/v2/permits/search'
HEADERS   = {'X-API-Key': API_KEY}

STATE     = 'IL'
DATE_FROM = '2014-01-01'
DATE_TO   = '2021-12-31'
PAGE_SIZE = 100  # max allowed

In [ ]:
def fetch_all_permits(geo_id, permit_from, permit_to, permit_tags=None, permit_q=None, page_size=100):
    """Fetch all permits using cursor-based pagination."""
    params = {
        'geo_id':        geo_id,
        'permit_from':   permit_from,
        'permit_to':     permit_to,
        'property_type': 'residential',
        'size':          page_size,
        'include_count': 'true',
    }
    if permit_tags:
        params['permit_tags'] = permit_tags
    if permit_q:
        params['permit_q'] = permit_q

    all_records = []
    page = 0

    while True:
        response = requests.get(BASE_URL, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        items = data.get('items', [])
        all_records.extend(items)
        page += 1

        if page == 1:
            total = data.get('total_count')
            print(f"  Total (API estimate): {total}")

        print(f"  Page {page}: {len(items)} records (running total: {len(all_records)})")

        next_cursor = data.get('next_cursor')
        if not next_cursor:
            break

        params['cursor'] = next_cursor
        params.pop('include_count', None)
        time.sleep(0.2)

    return all_records

In [ ]:
# Search for electric panel permits across keyword variants (2020-2021 only to gauge volume)
PANEL_FROM = '2020-01-01'
PANEL_TO   = '2021-12-31'

panel_terms = [
    ('electric panel',    None),
    ('electrical panel',  None),
    ('panel upgrade',     None),
    ('panel replacement', None),
    ('service upgrade',   None),
    ('service panel',     None),
    ('meter upgrade',     None),
    ('load center',       None),
]

print(f"{'Keyword':<25} {'Tag':<12} {'Count':>8}")
print("-" * 48)
for term, tag in panel_terms:
    params = {
        'geo_id': STATE, 'permit_from': PANEL_FROM, 'permit_to': PANEL_TO,
        'permit_q': term, 'property_type': 'residential',
        'size': 1, 'include_count': 'true',
    }
    if tag:
        params['permit_tags'] = tag
    r = requests.get(BASE_URL, headers=HEADERS, params=params)
    r.raise_for_status()
    total = r.json().get('total_count', {})
    count = total.get('value', 0) if isinstance(total, dict) else total
    relation = total.get('relation', 'eq') if isinstance(total, dict) else 'eq'
    suffix = '+' if relation == 'gte' else ''
    tag_label = tag or '(any)'
    print(f"{term:<25} {tag_label:<12} {str(count) + suffix:>8}")
    time.sleep(0.3)

In [ ]:
# Fetch heat pump permits using multiple keyword variants, then deduplicate
HP_KEYWORDS = ['heat pump', 'mini split', 'mini-split', 'minisplit', 'geothermal', 'ground source']

all_records = []
for term in HP_KEYWORDS:
    print(f"\n--- Fetching: '{term}' ---")
    records = fetch_all_permits(
        geo_id=STATE,
        permit_from=DATE_FROM,
        permit_to=DATE_TO,
        permit_tags='hvac',
        permit_q=term,
    )
    all_records.extend(records)

hp_df = pd.json_normalize(all_records).drop_duplicates(subset='id')
print(f"\nTotal unique heat pump permits: {len(hp_df):,}")

In [ ]:
permits = hp_df.copy()
print(f"Total heat pump permits: {len(permits):,}")
permits.head()

In [ ]:
# Fetch all panel-related permits 2020-2025, deduplicated
PANEL_FROM = '2020-01-01'
PANEL_TO   = '2025-12-31'

PANEL_KEYWORDS = [
    'electric panel',
    'electrical panel',
    'panel upgrade',
    'panel replacement',
    'service upgrade',
    'service panel',
    'meter upgrade',
    'load center',
]

panel_records = []
for term in PANEL_KEYWORDS:
    print(f"Fetching: '{term}'")
    records = fetch_all_permits(
        geo_id=STATE,
        permit_from=PANEL_FROM,
        permit_to=PANEL_TO,
        permit_tags=None,
        permit_q=term,
    )
    panel_records.extend(records)
    time.sleep(0.3)

panel_df = pd.json_normalize(panel_records).drop_duplicates(subset='id')
print(f"\nTotal unique panel-related permits (2020–2025): {len(panel_df):,}")
panel_df.head()

In [13]:
# Filter to residential + final, then analyze approval_duration
res_final = panel_df[
    (panel_df['property_type'] == 'residential') &
    (panel_df['status'] == 'final')
].copy()

print(f"Residential + final permits: {len(res_final):,}")

# Flag missing/invalid approval_duration
missing = res_final['approval_duration'].isna() | (res_final['approval_duration'] < 0)
print(f"Missing or invalid approval_duration: {missing.sum():,} ({missing.mean()*100:.1f}%)")

valid = res_final[~missing]['approval_duration']
print(f"\nApproval duration (days) — n={len(valid):,}")
print(f"  Mean:        {valid.mean():.1f}")
print(f"  Median:      {valid.median():.1f}")
print(f"  Same-day (0): {(valid == 0).sum():,} ({(valid == 0).mean()*100:.1f}%)")
print(f"  25th pct:    {valid.quantile(0.25):.1f}")
print(f"  75th pct:    {valid.quantile(0.75):.1f}")
print(f"  90th pct:    {valid.quantile(0.90):.1f}")

Residential + final permits: 1,407
Missing or invalid approval_duration: 40 (2.8%)

Approval duration (days) — n=1,367
  Mean:        11.4
  Median:      5.0
  Same-day (0): 489 (35.8%)
  25th pct:    0.0
  75th pct:    11.0
  90th pct:    27.0
